In [ ]:
import numpy as np
from .exceptions import SampleSizeExpendedError
from sklearn.linear_model import SGDRegressor


SERIALIZED_ATTRIBUTES = ["N", "D", "delta", "q", "intercept", "phi"]


class BWD(object):
    """**The Balancing Walk Design with Restarts**

    This is the primary suggested algorithm from [Arbour et al (2022)](https://arxiv.org/abs/2203.02025).
    At each step, it adjusts randomization probabilities to ensure that imbalance tends towards zero. In
    particular, if current imbalance is w and the current covariate profile is $x$, then the probability of
    treatment conditional on history will be:

    $$p_i = q \\left(1 - \\phi \\frac{x \\cdot w}{\\alpha}\\right)$$

    $q$ is the desired marginal probability, $\\phi$ is the parameter which controls robustness and
    $\\alpha$ is the normalizing constant which ensures the probability is well-formed.

    !!! important "If $|x \\cdot w| > \\alpha$"
        A restart is performed by resetting the algorithm:

        - $w$ is reset to the zero vector
        - $\\alpha$ is reset to a constant based on the number of units remaining in the sample
    """

    def __init__(
        self,
        N: int,
        D: int,
        delta: float = 0.05,
        q: float = 0.5,
        intercept: bool = True,
        phi: float = 1,
    ) -> None:
        """
        Args:
            N: total number of points
            D: dimension of the data
            delta: probability of failure
            q: Target marginal probability of treatment
            intercept: Whether an intercept term be added to covariate profiles
            phi: Robustness parameter. A value of 1 focuses entirely on balance, while a value
                approaching zero does pure randomization.
        """
        self.q = q
        self.intercept = intercept
        self.delta = delta
        self.N = N
        self.D = D + int(self.intercept)
        self.value_plus = 2 * (1 - self.q)
        self.value_minus = -2 * self.q
        self.phi = phi
        self.reset()

    def set_alpha(self, N: int) -> None:
        """Set normalizing constant for remaining N units

        Args:
            N: Number of units remaining in the sample
        """
        if N < 0:
            raise SampleSizeExpendedError()
        self.alpha = np.log(2 * N / self.delta) * min(1 / self.q, 9.32)

    def assign_next(self, x: np.ndarray) -> np.ndarray:
        """Assign treatment to the next point

        Args:
            x: covariate profile of unit to assign treatment
        """
        if self.intercept:
            x = np.concatenate(([1], x))
        dot = x @ self.w_i
        if abs(dot) > self.alpha:
            self.w_i = np.zeros((self.D,))
            self.set_alpha(self.N - self.iterations)
            dot = x @ self.w_i

        p_i = self.q * (1 - self.phi * dot / self.alpha)

        if np.random.rand() < p_i:
            value = self.value_plus
            assignment = 1
        else:
            value = self.value_minus
            assignment = -1
        self.w_i += value * x
        self.iterations += 1
        return int((assignment + 1) / 2)

    def assign_all(self, X: np.ndarray) -> np.ndarray:
        """Assign all points

        This assigns units to treatment in the offline setting in which all covariate
        profiles are available prior to assignment. The algorithm assigns as if units
        were still only observed in a stream.

        Args:
            X: array of size n × d of covariate profiles
        """
        return np.array([self.assign_next(X[i, :]) for i in range(X.shape[0])])

    @property
    def definition(self):
        return {
            "N": self.N,
            "D": self.D,
            "delta": self.delta,
            "q": self.q,
            "intercept": self.intercept,
            "phi": self.phi,
        }

    @property
    def state(self):
        return {"w_i": self.w_i, "iterations": self.iterations}

    def update_state(self, w_i, iterations, **model_state):
        self.w_i = np.array(w_i)
        self.iterations = iterations

    def reset(self):
        self.w_i = np.zeros((self.D,))
        self.set_alpha(self.N)
        self.iterations = 0

In [ ]:
class OutcomeBWD(BWD):
    def __init__(
        self,
        N: int,
        D: int,
        delta: float = 0.05,
        q: float = 0.5,
        intercept: bool = True,
        phi: float = 1,
        reg_strength: float = 0.0001
    ):
        super(OutcomeBWD).__init__(N, D, delta, q, intercept, phi)
        self.reg_strength = reg_strength
        self.model = SGDRegressor(alpha = self.reg_strength)

    def assign_next(self, x):
        super(OutcomeBWD, self).assign_next(self.model.predict(x))

    def partial_fit(self, a, x, y):
        if a == 0:
          self.model.partial_fit(x, y)

    @property
    def definition(self):
        return {
            "N": self.N,
            "D": self.D,
            "delta": self.delta,
            "q": self.q,
            "intercept": self.intercept,
            "phi": self.phi,
        } | self.model.definition

    @property
    def state(self):
        return {"w_i": self.w_i, "iterations": self.iterations} | self.model.state()

    def update_state(self, w_i, iterations, **model_state):
        self.w_i = np.array(w_i)
        self.iterations = iterations
        self.model.update_state(**model_state)

    def reset(self):
        self.w_i = np.zeros((self.D,))
        self.set_alpha(self.N)
        self.iterations = 0
        self.model.reset()

In [ ]:
from bwd import BWD
from numpy.random import default_rng
import numpy as np
rng = default_rng(2022)

n = 10000
d = 5
ate = 1
beta = rng.normal(size = d)

X = rng.normal(size = (n, d))

balancer = OutcomeBWD(N = n, D = d)
A_bwd = []
A_rand = []
imbalance_bwd = np.array([[0] * d])
imbalance_rand = np.array([[0] * d])

increment_imbalance = lambda imba, a, x: np.concatenate([imba, imba[-1:, :] + (2 * a - 1) * x])

for x in X:
    # Assign with BWD
    a_bwd = balancer.assign_next(x)
    imbalance_bwd = increment_imbalance(imbalance_bwd, a_bwd, x)
    A_bwd.append(a_bwd)

    # Assign with Bernoulli randomization
    a_rand = rng.binomial(n = 1, p = 0.5, size = 1).item()
    imbalance_rand = increment_imbalance(imbalance_rand, a_rand, x)
    A_rand.append(a_rand)

    y = X @ beta 

    balancer

# Outcomes are only realized at the conclusion of the experiment
eps = rng.normal(size=n)
Y_bwd = X @ beta + A_bwd * ate + eps
Y_rand = X @ beta + A_rand + ate + eps